## Stages 1,2,3 and 5

### Stages
1. Data Acquisition
2. Audio Engineering
3. Sentence Generation
4. TTS Generation 
5. Quality Control

Bash Script for running Stage 4 is isolated to a different runtime(see Colab_sateg4.ipynb) because of a conflicating version of transformers. Indic F5 requires trasnformer version -4.49.0

Note: 
- Each stage is resumable and cached in case if the colab session drops abruptly.
- You can also run stages in correct order but in differnt colab sessions. 
- All progress of each stage are cached in the mounted drive-storage. 

## Step 1 - Mount Google Drive 

Run the below cell and follow instruction on screen

In [1]:
# Mount google Drive
import os
from google.colab import drive
drive.mount('/content/drive')

# The code Repo, Clone/pull 
REPO_URL = 'https://github.com/rahulkolayikkath/synthetic-data-pipeline.git'  
%cd /content
if not os.path.exists('/content/synthetic-data-pipeline'):
    !git clone $REPO_URL
%cd /content/synthetic-data-pipeline
!git pull --ff-only

# Code import to working dir
!pip install -q -e .

# OUT_Dir  
OUT = '/content/drive/MyDrive/indic_synth/out_representative_sample'
os.makedirs(OUT, exist_ok=True)

Mounted at /content/drive
/content
Cloning into 'synthetic-data-pipeline'...
remote: Enumerating objects: 391, done.
remote: Counting objects: 100% (391/391), done.
remote: Compressing objects: 100% (255/255), done.
remote: Total 391 (delta 175), reused 328 (delta 112), pack-reused 0 (from 0)
Receiving objects: 100% (391/391), 5.55 MiB | 13.42 MiB/s, done.
Resolving deltas: 100% (175/175), done.
/content/synthetic-data-pipeline
Already up to date.
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for indic-synth (pyproject.toml) ... done


## Step 2 - Hugging Face Access 

Create a Hugging face token with read access
Make sure you have access to the following gated modelsa and datasets. Accept their terms on hugging face before you use them. 
1. ai4bharat/Kathbath 
2. ai4bharat/IndicF5
3. ai4bharat/indic-conformer-600m-multilingual
4. google/gemma-3-12b-it
5. sentence-transformers/LaBSE
6. speechbrain/spkrec-ecapa-voxceleb

In [2]:
# HF token
from getpass import getpass
from huggingface_hub import login
os.environ['HF_TOKEN'] = getpass('HF token:')
login(os.environ['HF_TOKEN'])

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


In [3]:
# Add HF Cache for Gemma(24gb) to avoid redownload on re-runs 
import os
os.environ["HF_HOME"] = "/content/hf_cache"

## Step 3 - Run stages

Config deatils 
1. config.representative.yaml 
- 3 languages x 2 speakers = 6 distinct speakers (gender-balanced)
- 3 languages x 1 topic x 4 sentance-types  = 12 Sentence bucket
- 8 sentances from each bucket, 8 x 12 = 96 sentences

In [4]:
# install requirements.txt
!pip install -r requirements.txt

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 16.1 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.7/4.7 MB 143.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 108.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.3/40.3 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 277.0/277.0 MB 1.6 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 314.2/314.2 kB 35.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 108.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.7/7.7 MB 105.1 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.6/121.6 kB 16.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.1/121.1 kB 16.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 788.2/788.2 kB 47.8 MB/s eta 0:00:00


In [5]:
!python scripts/run.py --config config.representative.yaml --stages data_acquisition

20:42:12 INFO    run | Pipeline start | out_dir=/content/drive/MyDrive/indic_synth/out_representative_sample seed=1234
20:42:12 INFO    run | =========== stage: data_acquisition ===========
20:42:12 INFO    run | Acquisition config: {'source': 'hf', 'repo_id': 'ai4bharat/Kathbath', 'local_dir': 'fake_kathbath', 'languages': ['hindi', 'malayalam', 'tamil'], 'split': 'valid', 'speakers_per_language': 2, 'clips_per_speaker': 2, 'min_total_speakers': 4, 'gender_balance': True, 'ref_min_dur': 3.0, 'ref_max_dur': 15.0, 'seed': 1234, 'out_dir': '/content/drive/MyDrive/indic_synth/out_representative_sample', 'hf_token': True, 'max_retries': 4, 'retry_backoff': 2.0, 'force_catalog': False}
20:42:13 INFO    run | [hindi] cataloging 2 parquet file(s) (audio column skipped)
20:42:38 INFO    run | [hindi] valid-00000-of-00002.parquet -> 1576 rows
20:43:04 INFO    run | [hindi] valid-00001-of-00002.parquet -> 1575 rows
20:43:04 INFO    run | [malayalam] cataloging 2 parquet file(s) (audio column ski

In [6]:
!python scripts/run.py --config config.representative.yaml --stages audio_engineering

20:44:46 INFO    run | Pipeline start | out_dir=/content/drive/MyDrive/indic_synth/out_representative_sample seed=1234
20:44:46 INFO    run | =========== stage: audio_engineering ===========
20:44:46 INFO    run | 12 downloaded clips, 0 already prepared, 12 to process
20:44:46 INFO    run | Prepare complete: {"stage": "audio_engineering", "elapsed_sec": 0.36, "target_sr": 24000, "norm": "peak", "trim": false, "prepared": 12, "failed": 0, "sr_in": {"16000": 12}, "flags": {"resampled_up": 12}, "backends": {"soundfile": 12}, "prepared_manifest": "/content/drive/MyDrive/indic_synth/out_representative_sample/prepared_manifest.jsonl"}
20:44:46 INFO    run | Requested stages in pipeline done in 0.4s.


In [7]:
!python scripts/run.py --config config.representative.yaml --stages sentence_generation

20:44:53 INFO    run | Pipeline start | out_dir=/content/drive/MyDrive/indic_synth/out_representative_sample seed=1234
20:44:53 INFO    run | =========== stage: sentence_generation ===========
processor_config.json: 100% 70.0/70.0 [00:00<00:00, 302kB/s]
chat_template.json: 100% 1.61k/1.61k [00:00<00:00, 4.86MB/s]
preprocessor_config.json: 100% 570/570 [00:00<00:00, 3.05MB/s]
config.json: 100% 916/916 [00:00<00:00, 4.67MB/s]
tokenizer_config.json: 100% 1.16M/1.16M [00:00<00:00, 14.9MB/s]
tokenizer.json: 100% 33.4M/33.4M [00:01<00:00, 25.7MB/s]
added_tokens.json: 100% 35.0/35.0 [00:00<00:00, 74.4kB/s]
special_tokens_map.json: 100% 662/662 [00:00<00:00, 2.63MB/s]
model.safetensors.index.json: 100% 109k/109k [00:00<00:00, 22.6MB/s]
Fetching 5 files: 100% 5/5 [15:05<00:00, 181.07s/it]
Download complete: 100% 24.4G/24.4G [15:05<00:00, 26.9MB/s]                
Loading weights:   0% 1/1065 [00:08<2:27:36,  8.32s/it]/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213:

In [ ]:
# STOP HERE and Run stage 4 on colab_stage4.ipynb

In [ ]:
!python scripts/run.py --config config.representative.yaml --stages quality_control